# ML code

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression


from scipy.optimize import minimize


# Column name mapping

In [3]:
def mapping_column_name(df, case='any', mapping_df=None, verbosity=1):
    if mapping_df is None:
        default_column_name_mapping_file_path = 'column_name_mapping.csv'
        try:
            mapping_df=pd.read_csv(default_column_name_mapping_file_path)
        except:
            raise Exception(f'The default column name file for mapping "{default_column_name_mapping_file_path}" was not found')

    mapping_df = mapping_df.query('use_case == @case')
    if len(mapping_df) == 0:
        raise Exception(f'The mapping dataframe does not contains the case: {case}')
    
    df_mapped = df.copy()
    columns_to_map = df.columns 
    if len(columns_to_map) != len(set(columns_to_map)):
        unique_columns = set(columns_to_map)
        for item in unique_columns:
            columns_to_map.to_list().remove(item)
        raise Exception(f'The column names should be unique but duplicate(s) are found: {set(columns_to_map)}')

    mapping_dictionary = dict()
    for col in columns_to_map:
        mask = mapping_df.apply(lambda row: row.astype(str).str.fullmatch(col, case=False, na=False).any(), axis=1)
        mapping_found = mapping_df[mask]
        if len(mapping_found) == 1:
            df_mapped.rename(columns={col : mapping_found['standard_name'].astype(str).iloc[0]}, inplace=True)
            if verbosity > 0: print(f'The column {col} was renamed to {mapping_found["standard_name"].iloc[0]}')
            mapping_dictionary[col] = mapping_found['standard_name'].astype(str).iloc[0]
        elif len(mapping_found) > 1:
            raise Exception(f'The column {col} has multiple mapping toward the following standard name: {mapping_found["standard_name"].to_list()}')
        elif len(mapping_found) == 0:
            mapping_dictionary[col] = col
            if verbosity > 1: print(f'The column {col} has no mapping found')
  
    return df_mapped, mapping_dictionary

df = pd.DataFrame([[2, 2.1, 2.2], [3, 3.1, 3.2]], columns=['Cycle', 'TiCl4_Pulse', 'TiCl4_Purge'])
df, _ = mapping_column_name(df,)
df

The column Cycle was renamed to cycle
The column TiCl4_Pulse was renamed to ticl4_pulse_time
The column TiCl4_Purge was renamed to ticl4_purge_time


,cycle,ticl4_pulse_time,ticl4_purge_time
0,2,2.1,2.2
1,3,3.1,3.2


# Create polynomial features for multiple layer process

In [4]:
df = pd.DataFrame([[2, 2.1, 2.2, 0.01], [3, 3.1, 3.2, 0.07]], columns=['Cycle', 'TiCl4_Pulse', 'Temperature', 'Vfb'])
df, _ = mapping_column_name(df)

col_for_poly_feature = df.columns.to_list()
col_for_poly_feature.remove('vfb')

print(col_for_poly_feature)
poly_feature_creator = PolynomialFeatures(degree=2,interaction_only=True, include_bias=False)
poly_feature_creator.fit(df[col_for_poly_feature])

poly_feature = pd.DataFrame(poly_feature_creator.transform(df[col_for_poly_feature]), columns=poly_feature_creator.get_feature_names_out())
poly_feature
# pd.DataFrame(

The column Cycle was renamed to cycle
The column TiCl4_Pulse was renamed to ticl4_pulse_time
The column Temperature was renamed to temperature
The column Vfb was renamed to vfb
['cycle', 'ticl4_pulse_time', 'temperature']


,cycle,ticl4_pulse_time,temperature,cycle ticl4_pulse_time,cycle temperature,ticl4_pulse_time temperature
0,2.0,2.1,2.2,4.2,4.4,4.62
1,3.0,3.1,3.2,9.3,9.6,9.92


In [ ]:
# search with randomized starts
solutions = []

def objective(x):
    pred = model.predict([x])[0]
    return abs(pred - y_target)


for i in range(50000):
    # random starts as initial guess
    temp = np.random.uniform(80,150)
    time = np.random.uniform(1,10)
    
    result = minimize(objective, x0=[temp,time], bounds=[(80,150),(1,10)])
    result.x # the optimal input
    result.fun # the minimum objective value found

    if abs(result.fun) < 0.01:
        solutions.append([result.fun, result.fun+y_target,  temp,time])

# Undirected Graph

# Performance Plot

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error

def get_metrics_text(y_true, y_pred):
    """【輔助函數】計算指標並回傳格式化字串"""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    # Matplotlib 支援用 $ 包起來呈現數學斜體 (LaTeX 風格)
    return f"MAE: {mae:.2f}\nRMSE: {rmse:.2f}\nR²: {r2:.2f}\nMAPE: {mape * 100:.2f}%"

def plot_comprehensive_performance(y_train_true, y_train_pred, y_test_true, y_test_pred, save_path=None):
    """
    繪製 3x2 的迴歸模型完整評估圖表。
    若有提供 save_path (例如 'report.png')，則會自動儲存高畫質圖片。
    """
    sns.set_theme(style="whitegrid")
    fig, axes = plt.subplots(3, 2, figsize=(15, 18))
    fig.suptitle('Regression Performance: Train vs Test', fontsize=20, fontweight='bold', y=1.02)

    # ==========================================
    # 內部輔助函數：大幅減少重複的繪圖程式碼
    # ==========================================
    def draw_scatter(ax, yt, yp, title, color, label=None, alpha=0.5):
        sns.scatterplot(x=yt, y=yp, ax=ax, color=color, alpha=alpha, label=label)
        # 動態計算範圍並畫出 y=x 的完美預測紅虛線
        min_val, max_val = min(np.min(yt), np.min(yp)), max(np.max(yt), np.max(yp))
        ax.plot([min_val, max_val], [min_val, max_val], color='crimson', linestyle='--', linewidth=2)
        ax.set(title=title, xlabel='Actual Values', ylabel='Predicted Values')

    def draw_hist(ax, yt, yp, title, color, label=None, alpha=0.5, stat='count'):
        sns.histplot(yt - yp, kde=True, ax=ax, color=color, bins=30, alpha=alpha, label=label, stat=stat)
        ax.axvline(x=0, color='crimson', linestyle='--', linewidth=2)
        ax.set(title=title, xlabel='Residuals (Actual - Predicted)', ylabel='Frequency/Density')

    # ==========================================
    # Row 1 & Row 2: 獨立繪製 Train 與 Test
    # ==========================================
    datasets = [
        (0, 'Train Set', y_train_true, y_train_pred, 'royalblue', 'teal'),
        (1, 'Test Set',  y_test_true,  y_test_pred,  'darkorange', 'indianred')
    ]

    for row, name, yt, yp, color_scatter, color_hist in datasets:
        # 1. 取得指標字串並在終端機印出
        metrics = get_metrics_text(yt, yp)
        print(f"=== {name} ===\n{metrics}\n")
        
        # 2. 畫散佈圖與文字方塊
        draw_scatter(axes[row, 0], yt, yp, f'{name}: Predicted vs Actual', color_scatter)
        axes[row, 0].text(0.05, 0.95, metrics, transform=axes[row, 0].transAxes, 
                          fontsize=11, va='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        # 3. 畫殘差直方圖
        draw_hist(axes[row, 1], yt, yp, f'{name}: Residuals', color_hist)

    # ==========================================
    # Row 3: 疊加對比 (Combined Overlay)
    # ==========================================
    # 畫疊加散佈圖
    draw_scatter(axes[2, 0], y_train_true, y_train_pred, 'COMBINED: Predicted vs Actual', 'royalblue', 'Train', 0.3)
    draw_scatter(axes[2, 0], y_test_true, y_test_pred, 'COMBINED: Predicted vs Actual', 'darkorange', 'Test', 0.6)
    axes[2, 0].legend()

    # 畫疊加殘差圖 (注意這裡 stat='density' 才能公平比較不同資料量的形狀)
    draw_hist(axes[2, 1], y_train_true, y_train_pred, 'COMBINED: Residuals Density', 'teal', 'Train', 0.3, stat='density')
    draw_hist(axes[2, 1], y_test_true, y_test_pred, 'COMBINED: Residuals Density', 'indianred', 'Test', 0.5, stat='density')
    axes[2, 1].legend()

    plt.tight_layout()
    
    # ==========================================
    # 儲存圖片邏輯 (dpi=300 確保高解析度，bbox_inches='tight' 避免邊緣被裁切)
    # ==========================================
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"圖表已成功儲存至: {save_path}")
        
    plt.show()

# --- 測試與使用範例 ---
# 假設你已經有模型預測結果了：
# plot_comprehensive_performance(y_train_true, y_train_pred, y_test_true, y_test_pred, save_path="model_v1_evaluation.png")